# ERA5 to PRISM Downscaling (Georgia)

Spatiotemporal regional downscaling pipeline using temporal ERA5 inputs and PRISM targets.


## 1. Objective
Estimate high-resolution PRISM temperature fields from coarse ERA5 reanalysis context.

Formulation: `ERA5(t-k+1 ... t) -> PRISM(t)`

Temporal context is essential because local temperature structure depends on recent evolution, not only the latest frame.


## 2. Data
- **ERA5**: coarse-resolution reanalysis input sequence
- **PRISM**: higher-resolution daily raster target

The dataset aligns dates, clips/reprojects rasters, and constructs temporal windows with configurable history length.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display, Markdown

ROOT = Path.cwd().resolve()
if not (ROOT / 'datasets').exists() and (ROOT.parent / 'datasets').exists():
    ROOT = ROOT.parent

ERA5_PATH = ROOT / 'data_raw/era5_georgia_temp.nc'
PRISM_DIR = ROOT / 'data_raw/prism'
RESULTS_DIR = ROOT / 'results/evaluation'

print(f'Project root: {ROOT}')
print(f'ERA5 available: {ERA5_PATH.exists()}')
print(f'PRISM available: {PRISM_DIR.exists()}')
print(f'Results directory available: {RESULTS_DIR.exists()}')


## 3. Model Progression
1. Persistence baseline
2. Global linear baseline
3. Spatial CNN baseline
4. ConvLSTM temporal model (main model)

This progression isolates the contribution of temporal modeling over simpler baselines.


In [ ]:
if ERA5_PATH.exists() and PRISM_DIR.exists():
    try:
        from datasets.prism_dataset import ERA5_PRISM_Dataset
        ds = ERA5_PRISM_Dataset(str(ERA5_PATH), str(PRISM_DIR), history_length=5, verbose=False)
        x, y = ds[0]
        print(f'Aligned samples: {len(ds)}')
        print(f'Input shape [T,C,H,W]: {tuple(x.shape)}')
        print(f'Target shape [C,H,W]: {tuple(y.shape)}')
    except Exception as exc:
        print(f'Dataset construction failed: {exc}')
else:
    print('Missing data. Run:')
    print('python data_pipeline/download_era5_georgia.py --year 2023 --month 1')
    print('python data_pipeline/download_prism.py --start-date 20230101 --days 20 --variable tmean')


## 4. Results
Loads saved evaluation artifacts from `results/evaluation/`.


In [ ]:
cnn_imgs = sorted((RESULTS_DIR / 'cnn').glob('comparison_*.png'))
convlstm_imgs = sorted((RESULTS_DIR / 'convlstm').glob('comparison_*.png'))
cnn_metrics = RESULTS_DIR / 'cnn' / 'metrics.json'
convlstm_metrics = RESULTS_DIR / 'convlstm' / 'metrics.json'
summary_csv = RESULTS_DIR / 'baselines_summary.csv'
comparison_fig = RESULTS_DIR / 'model_comparison.png'

if cnn_imgs:
    display(Markdown('### CNN Result'))
    display(Image(filename=str(cnn_imgs[0])))
else:
    display(Markdown('CNN comparison image not found.'))

if convlstm_imgs:
    display(Markdown('### ConvLSTM Result'))
    display(Image(filename=str(convlstm_imgs[0])))
else:
    display(Markdown('ConvLSTM comparison image not found.'))

if comparison_fig.exists():
    display(Markdown('### Cross-Model Comparison'))
    display(Image(filename=str(comparison_fig)))

rows = []
for name, path in [('cnn', cnn_metrics), ('convlstm', convlstm_metrics)]:
    if path.exists():
        data = json.loads(path.read_text())
        rows.append({
            'model': name,
            'rmse': data.get('rmse'),
            'mae': data.get('mae'),
            'bias': data.get('bias'),
            'num_samples': data.get('num_samples'),
        })

if rows:
    display(Markdown('### Metrics (JSON)'))
    display(pd.DataFrame(rows))

if summary_csv.exists():
    display(Markdown('### Baselines Summary'))
    display(pd.read_csv(summary_csv))

if not (cnn_imgs or convlstm_imgs or summary_csv.exists()):
    display(Markdown('Run evaluation to generate outputs:'))
    print('python training/train_downscaler.py --model cnn --history-length 5 --epochs 20 --batch-size 4 --learning-rate 1e-3')
    print('python training/train_downscaler.py --model convlstm --history-length 5 --epochs 20 --batch-size 4 --learning-rate 1e-3')
    print('python evaluation/evaluate_model.py --models persistence linear cnn convlstm --history-length 5 --num-samples 8')


## 5. Interpretation
Temporal modeling can improve over purely spatial baselines by using sequence context, but performance remains constrained by limited data span, variable set, and coarse-to-fine resolution mismatch.


## 6. Relation to Published Directions
Large weather-climate systems (e.g., FourCastNet, GraphCast, Prithvi WxC) motivate this direction, but this repository focuses on a tractable regional pipeline. The implemented contribution is a robust model progression from classical baselines to a ConvLSTM temporal model with reproducible evaluation outputs.
